# AutoEncoders

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import random
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST
import pandas as pd


In [18]:
torch.manual_seed(0)
torch.cuda.manual_seed(0)
np.random.seed(0)
random.seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [21]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = MNIST(root='./data', train=False, download=True, transform=transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 495kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.61MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.3MB/s]


In [19]:
class AutoEncoder(nn.Module):
  def __init__(self, latent_dim=32):
    super().__init__()
    self.encoder = nn.Sequential(
        nn.Linear(28*28, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, latent_dim)
    )

    self.decoder = nn.Sequential(
        nn.Linear(latent_dim, 64),
        nn.ReLU(),
        nn.Linear(64, 128),
        nn.ReLU(),
        nn.Linear(128, 28*28),
        nn.Sigmoid()
    )

    def forward(self, x):
      x = x.flatten(start_dim=1)
      encoded = self.encoder(x)
      decoded = self.decoder(encoded)
      return encoded, decoded


In [27]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

for img, lab in train_loader:
  print(img.shape)
  print(lab.shape)
  break

torch.Size([64, 1, 28, 28])
torch.Size([64])


In [30]:
# training Script
def train(model, train_loader, criterion, optimizer, t_iterations):
  print("Training started")
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model.to(device)

  criterion = nn.MSELoss()
  optimizer = optim.Adam(model.parameters(), lr=0.001)
  prog_bar = tqdm(range(t_iterations))

  counter = 0
  training = True

  train_loss = []
  batch_train_loss = []

  while training:
    for img, label in train_loader:
      img = img.to(device)
      label = label.to(device)

      encoded, decoded = model(img)
      loss = criterion(decoded, img)
      train_loss.append(loss.item())
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

      if counter % 100 == 0:
        batch_train_loss.append(np.mean(train_loss))
        train_loss = []
      counter += 1
      prog_bar.update(1)
      if counter >= t_iterations:
        print("Training Done")
        training = False
        break
  return batch_train_loss




In [31]:
def evaluate(model, test_loader, device):
  model.eval()

  eval_loss = []
  with torch.no_grad():
    for imgs, labels in test_loader:
      imgs = imgs.to(device)
      labels = labels.to(device)

      encoded, decoded = model(imgs)
      loss = criterion(decoded, imgs)
      eval_loss.append(loss.item())
  return np.mean(eval_loss)

In [ ]:
vanilla_model = AutoEncoder()

vanilla_model = train(vanilla_model, train_loader, criterion, optimizer, 10000)